# Hallucination-Resistant Framework Evaluation Results
This notebook visualizes the empirical performance of EdgeCore v1.0, Final Thought v1.1, and ULTIMA-X v2.1 based on our comprehensive evaluation suite.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

results_dir = 'results/'
# Ensure plots look nice
plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (10, 6)

## 1. EdgeCore v1.0: Ablation Study (Faithfulness Improvement)
We compare the baseline model against EdgeCore, which utilizes Dual Key-Value banks and Verification-Gated attention to adhere to injected facts.

In [ ]:
try:
    df_edge = pd.read_csv(os.path.join(results_dir, 'edgecore_ablation.csv'))
    
    # Calculate means
    means = df_edge.groupby('gating_active')['faithfulness_score'].mean()
    
    labels = ['Baseline (No Gating)', 'EdgeCore (Gating Active)']
    values = [means.get(False, 0), means.get(True, 0)]
    
    plt.figure(figsize=(8, 6))
    bars = plt.bar(labels, values, color=['#e74c3c', '#2ecc71'])
    
    plt.ylabel('Average Faithfulness Score (0.0 to 1.0)')
    plt.title('Impact of Dual KV Gating on Evidence Adherence')
    plt.ylim(0, 1.2)
    
    for bar in bars:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2, yval + 0.02, f'{yval:.2%}', ha='center', va='bottom', fontweight='bold')
        
    plt.show()
except Exception as e:
    print(f"Could not load or plot edgecore data: {e}")

## 2. Final Thought v1.1: Verification Latency & Cache Optimization
This section examines the latency introduced by asynchronous multi-claim verification, and how the LRU Execution Cache offsets this penalty.

In [ ]:
try:
    df_pipe = pd.read_csv(os.path.join(results_dir, 'pipeline_eval.csv'))
    
    # We'll plot Latency over sequential calls to show Cache warming
    plt.figure(figsize=(12, 5))
    plt.plot(df_pipe.index, df_pipe['latency_ms'], marker='o', linestyle='-', color='#3498db', linewidth=2)
    
    plt.axhline(y=df_pipe['latency_ms'].mean(), color='r', linestyle='--', label=f"Mean Latency: {df_pipe['latency_ms'].mean():.1f}ms")
    
    plt.xlabel('Query Sequence Number')
    plt.ylabel('Latency (ms)')
    plt.title('Pipeline Execution Latency per Query (Showing Cache Optimization)')
    plt.legend()
    plt.grid(True)
    plt.show()
    
    # Print some stats
    print(f"Total Queries Processed: {len(df_pipe)}")
    print(f"Average Badge Precision: {df_pipe['badge_precision'].mean():.2%}")
    
except Exception as e:
    print(f"Could not load or plot pipeline data: {e}")

## 3. Context-Aware v2.1: Long-Term Memory Persistence
Testing if the Episodic Memory Engine correctly retains facts and guards against contradictions after long distraction-filled sessions.

In [ ]:
try:
    df_mem = pd.read_csv(os.path.join(results_dir, 'memory_persistence.csv'))
    
    # Group by the number of session turns to see if length deteriorates recall
    summary = df_mem.groupby('num_turns').agg({
        'memory_retained_in_store': 'mean',
        'contradiction_caught': 'mean'
    }).reset_index()
    
    plt.figure(figsize=(10, 6))
    
    plt.plot(summary['num_turns'], summary['memory_retained_in_store'], marker='s', label='Storage Retention Rate', color='#2ecc71', linewidth=2.5, markersize=8)
    plt.plot(summary['num_turns'], summary['contradiction_caught'], marker='^', label='Contradiction Defense Rate', color='#e67e22', linewidth=2.5, markersize=8)
    
    plt.xlabel('Number of Distracting Session Turns')
    plt.ylabel('Success Rate (0 to 1)')
    plt.title('Memory Persistence & Contradiction Defense Over Extended Sessions')
    plt.ylim(-0.1, 1.1)
    plt.xticks(summary['num_turns'])
    plt.legend()
    plt.grid(True)
    plt.show()
    
except Exception as e:
    print(f"Could not load or plot memory data: {e}")